# Check the data: foreign keys, totals and dates

Do not take generated data on trust. This notebook generates a dataset and then tests it the way a careful reviewer would: every key matches, every total adds up, every date is in order. The checks are plain SQL, so you can reuse them on any dataset.

**Uses:** 5,000 records, no AI-generated fields. API reference: [generaterealdata.com/docs/api/](https://www.generaterealdata.com/docs/api/?ref=github).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://www.generaterealdata.com?ref=github), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://www.generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. Generate a dataset to check

In [2]:
def repo_file(path):
    """Read a file of this repository: from disk when the repo is cloned, from GitHub otherwise (e.g. in Colab)."""
    local = Path("..") / path
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get("https://raw.githubusercontent.com/generaterealdata/api-examples/main/" + path, timeout=30)
    r.raise_for_status()
    return r.text


SHOP_SQL = repo_file("schemas/shop.sql")
SHOP_RULES = repo_file("schemas/shop_rules.txt").strip()

ds = generate({"mode": "sql", "sql": SHOP_SQL, "notes": SHOP_RULES, "rows": 5000,
               "format": "sql", "dialect": "postgresql", "seed": 6})
summary(ds)

{'status': 'ready',
 'format': 'sql',
 'rows_generated': 5000,
 'rows_per_table': {'products': 89,
  'order_items': 3126,
  'customers': 446,
  'orders': 1339},
 'credits_charged': 5000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_VXOewGLlsQvdxAKi',
 'seed': 6}

In [3]:
import sqlite3

import pandas as pd

con = sqlite3.connect(download(ds, "shop_to_check.sqlite", file="db"))


def count(sql):
    return con.execute(sql).fetchone()[0] or 0

## 2. Checks that work on any dataset

These read the table definitions from the database itself, so they need no knowledge of the schema: foreign keys that point nowhere, duplicate primary keys, duplicates in `UNIQUE` columns, and empty values in `NOT NULL` columns.

In [4]:
tables = [t for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'")]
results = []
for table in tables:
    columns = con.execute(f'PRAGMA table_info("{table}")').fetchall()          # (cid, name, type, notnull, default, pk)
    for fk in con.execute(f'PRAGMA foreign_key_list("{table}")').fetchall():                # (id, seq, parent, from, to, ...)
        parent, child_column, parent_column = fk[2], fk[3], fk[4]
        orphans = count(f'SELECT COUNT(*) FROM "{table}" c LEFT JOIN "{parent}" p ON p."{parent_column}" = c."{child_column}" '
                        f'WHERE c."{child_column}" IS NOT NULL AND p."{parent_column}" IS NULL')
        results.append((f"{table}.{child_column} exists in {parent}", orphans))
    for _, name, _, notnull, _, pk in columns:
        if pk:
            results.append((f"{table}.{name} is unique (primary key)", count(f'SELECT COUNT(*) - COUNT(DISTINCT "{name}") FROM "{table}"')))
        if notnull:
            results.append((f"{table}.{name} is never empty", count(f'SELECT COUNT(*) FROM "{table}" WHERE "{name}" IS NULL')))
    for index in con.execute(f'PRAGMA index_list("{table}")').fetchall():       # UNIQUE constraints
        if index[2]:
            cols = ", ".join(f'"{r[2]}"' for r in con.execute(f'PRAGMA index_info("{index[1]}")'))
            results.append((f"{table} ({cols.replace(chr(34), '')}) is unique",
                            count(f'SELECT COUNT(*) FROM (SELECT 1 FROM "{table}" GROUP BY {cols} HAVING COUNT(*) > 1)')))
structure = pd.DataFrame(results, columns=["check", "rows breaking it"])
print(f"{len(structure)} structural checks, {int((structure['rows breaking it'] > 0).sum())} with problems")
structure[structure["check"].str.contains("exists in|unique")]

24 structural checks, 0 with problems


,check,rows breaking it
0,customers.customer_id is unique (primary key),0
5,customers (email) is unique,0
6,products.product_id is unique (primary key),0
10,orders.customer_id exists in customers,0
11,orders.order_id is unique (primary key),0
16,order_items.product_id exists in products,0
17,order_items.order_id exists in orders,0
18,order_items.order_item_id is unique (primary key),0


## 3. Checks for this dataset's rules

The schema's `CHECK` constraints and the rules sent with it, each as a query that counts the rows breaking it.

In [5]:
rules = {
    "every customer has at least one order":
        "SELECT COUNT(*) FROM customers c WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)",
    "every order has at least one item":
        "SELECT COUNT(*) FROM orders o WHERE NOT EXISTS (SELECT 1 FROM order_items i WHERE i.order_id = o.order_id)",
    "item price equals the product's price":
        "SELECT COUNT(*) FROM order_items i JOIN products p USING (product_id) WHERE ABS(i.unit_price - p.unit_price) > 0.005",
    "line_total = quantity x unit_price":
        "SELECT COUNT(*) FROM order_items WHERE ABS(line_total - quantity * unit_price) > 0.011",
    "order_total = sum of the order's line totals":
        "SELECT COUNT(*) FROM orders o JOIN (SELECT order_id, SUM(line_total) s FROM order_items GROUP BY order_id) x "
        "USING (order_id) WHERE ABS(o.order_total - x.s) > 0.011",
    "order date is never before the customer's signup":
        "SELECT COUNT(*) FROM orders o JOIN customers c USING (customer_id) WHERE o.order_date < c.signup_date",
    "order dates are inside 2025-01-01 to 2026-09-30":
        "SELECT COUNT(*) FROM orders WHERE order_date < '2025-01-01' OR order_date > '2026-09-30'",
    "status is one of the four allowed values":
        "SELECT COUNT(*) FROM orders WHERE status NOT IN ('pending', 'shipped', 'delivered', 'cancelled')",
    "category is one of the six allowed values":
        "SELECT COUNT(*) FROM products WHERE category NOT IN ('Electronics', 'Home', 'Kitchen', 'Toys', 'Sports', 'Books')",
    "prices are above zero":
        "SELECT COUNT(*) FROM products WHERE unit_price <= 0",
    "quantity is 1 to 5":
        "SELECT COUNT(*) FROM order_items WHERE quantity NOT BETWEEN 1 AND 5",
}
pd.DataFrame([(rule, count(sql)) for rule, sql in rules.items()], columns=["rule", "rows breaking it"])

,rule,rows breaking it
0,every customer has at least one order,0
1,every order has at least one item,0
2,item price equals the product's price,0
3,line_total = quantity x unit_price,0
4,order_total = sum of the order's line totals,0
5,order date is never before the customer's signup,0
6,order dates are inside 2025-01-01 to 2026-09-30,0
7,status is one of the four allowed values,0
8,category is one of the six allowed values,0
9,prices are above zero,0


## 4. Distributions

Rules about proportions cannot be right or wrong row by row; compare them with what was asked (delivered 70%, shipped 12%, pending 8%, cancelled 10%).

In [6]:
pd.read_sql_query("""
    SELECT status, COUNT(*) AS orders, ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 1) AS percent
    FROM orders GROUP BY status ORDER BY orders DESC""", con)

,status,orders,percent
0,delivered,911,68.0
1,shipped,159,11.9
2,cancelled,149,11.1
3,pending,120,9.0


## Reuse it

- Section 2 works unchanged on the SQLite file of any dataset.
- For your own rules, write one query per rule that counts the rows breaking it, and fail your pipeline if any count is above 0.
- The [free dataset library](https://github.com/generaterealdata/synthetic-datasets) publishes such a `checks.sql` with every dataset, with between 19 and 48 rules each.